In [1]:
import h2o
import numpy as np
import pandas as pd
import re
import joblib

#helper function
import helper

#scikit-learn libraries
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, roc_auc_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import KNNImputer

#h2o models and grid search
from h2o.estimators.gbm import H2OGradientBoostingEstimator
from h2o.estimators.deeplearning import H2ODeepLearningEstimator
from h2o.grid.grid_search import H2OGridSearch

#plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

#XGBoost
from xgboost import XGBClassifier

#model explainability
import shap

%matplotlib inline

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [4]:
df=pd.read_csv(r"D:\LoanLens\data\non_standardized_data.csv")

In [5]:
df.head()

,loan_id,customer_id,loan_status,current_loan_amount,term,credit_score,years_in_current_job,home_ownership,annual_income,purpose,monthly_debt,years_of_credit_history,months_since_last_delinquent,number_of_open_accounts,number_of_credit_problems,current_credit_balance,maximum_open_credit,bankruptcies,tax_liens
0,0000757f-a121-41ed-b17b-162e76647c1f,dde79588-12f0-4811-bab0-e2b07f633fcd,loan_given,11731.0,0,746.0,4.0,rent,50025.0,debt_consolidation,355.18,11.5,28.4,12,0,17391,37081.0,0.0,0.0
1,0000afa6-8902-4f8f-b870-25a8fdad0aeb,e49c1a82-a0f7-45e8-9f46-2f75c43f9fbc,loan_refused,24613.0,1,744.8,6.0,rent,95414.0,business_loan,542.29,17.6,73.0,7,0,14123,16954.0,0.0,0.0
2,00020fb0-6b8a-4b3a-8c72-9c4c847e8cb6,c9decd06-16f7-44c3-b007-8776f2a9233d,loan_given,19018.4,0,742.0,3.0,home_mortgage,64760.0,debt_consolidation,582.84,26.8,23.2,11,0,8880,22711.0,0.0,0.0
3,00045ecd-59e9-4752-ba0d-679ff71692b3,b7bce684-b4b0-4b29-af66-eae316bce573,loan_given,11863.0,0,734.0,10.0,own_home,69202.0,debt_consolidation,859.26,30.3,33.2,7,0,9959,16995.0,0.0,0.0
4,0004f37b-5859-40f6-98d0-367aa3b3f3f1,f662b062-5fa5-463d-b5c0-4e36d09fcab1,loan_given,13719.0,0,724.0,1.0,own_home,34297.0,home_improvements,777.38,13.6,2.0,12,0,6720,53335.0,0.0,0.0


In [6]:
df.drop(['loan_id', 'customer_id'], axis = 1, inplace = True)

In [7]:
df.purpose.value_counts()

purpose
debt_consolidation      70446
other                    7767
home_improvements        5205
business_loan            1328
buy_a_car                1196
medical_bills             955
buy_house                 559
take_a_trip               454
major_purchase            344
small_business            248
moving                    124
wedding                    99
educational_expenses       92
vacation                   82
renewable_energy            9
Name: count, dtype: int64

## REMOVING MULTICOLLINEARITY

In [10]:
purpose_mapping = {
    'debt_consolidation': 'debt_consolidation',
    'business_loan': 'business_loans',
    'small_business': 'business_loans',
    'other': 'other',
    'home_improvements': 'personal_loans',
    'buy_a_car': 'personal_loans',
    'medical_bills': 'personal_loans',
    'buy_house': 'personal_loans',
    'take_a_trip': 'personal_loans',
    'major_purchase': 'personal_loans',
    'moving': 'personal_loans',
    'wedding': 'personal_loans',
    'educational_expenses': 'personal_loans',
    'vacation': 'personal_loans',
    'renewable_energy': 'personal_loans',
}

purpose_filename = 'D:\LoanLens\model\purpose_mapping.pkl'
joblib.dump(purpose_mapping, purpose_filename)

df.replace({"purpose": purpose_mapping}, inplace=True)

df.purpose.value_counts()

purpose
debt_consolidation    70446
personal_loans         9119
other                  7767
business_loans         1576
Name: count, dtype: int64

In [11]:
df.head()

,loan_status,current_loan_amount,term,credit_score,years_in_current_job,home_ownership,annual_income,purpose,monthly_debt,years_of_credit_history,months_since_last_delinquent,number_of_open_accounts,number_of_credit_problems,current_credit_balance,maximum_open_credit,bankruptcies,tax_liens
0,loan_given,11731.0,0,746.0,4.0,rent,50025.0,debt_consolidation,355.18,11.5,28.4,12,0,17391,37081.0,0.0,0.0
1,loan_refused,24613.0,1,744.8,6.0,rent,95414.0,business_loans,542.29,17.6,73.0,7,0,14123,16954.0,0.0,0.0
2,loan_given,19018.4,0,742.0,3.0,home_mortgage,64760.0,debt_consolidation,582.84,26.8,23.2,11,0,8880,22711.0,0.0,0.0
3,loan_given,11863.0,0,734.0,10.0,own_home,69202.0,debt_consolidation,859.26,30.3,33.2,7,0,9959,16995.0,0.0,0.0
4,loan_given,13719.0,0,724.0,1.0,own_home,34297.0,personal_loans,777.38,13.6,2.0,12,0,6720,53335.0,0.0,0.0
